# Exploratory Data Analysis — Product Recommendation System
Exploring four raw datasets: `category_tree.csv`, `events.csv`, `item_properties_part1.csv`, and `item_properties_part2.csv`.

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

%matplotlib inline
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

## 2. Load CSV Files

In [ ]:
DATA_DIR = "../data/raw"

category_tree    = pd.read_csv(f"{DATA_DIR}/category_tree.csv")
events           = pd.read_csv(f"{DATA_DIR}/events.csv")
item_props_part1 = pd.read_csv(f"{DATA_DIR}/item_properties_part1.csv")
item_props_part2 = pd.read_csv(f"{DATA_DIR}/item_properties_part2.csv")

print("Loaded all 4 datasets successfully.")

## 3. DataFrame Overview

In [ ]:
datasets = {
    "category_tree":    category_tree,
    "events":           events,
    "item_props_part1": item_props_part1,
    "item_props_part2": item_props_part2,
}

for name, df in datasets.items():
    print(f"\n{'='*50}")
    print(f"  {name}  —  shape: {df.shape}")
    print(f"{'='*50}")
    display(df.head(3))
    print()
    df.info()
    print()
    display(df.describe(include="all"))

## 4. Explore Category Tree

In [ ]:
print(f"Total categories  : {category_tree['categoryid'].nunique()}")
print(f"Total rows        : {len(category_tree)}")
print(f"Null parent IDs   : {category_tree['parentid'].isna().sum()}  (these are root nodes)")

# Count children per parent
children_per_parent = category_tree.groupby("parentid")["categoryid"].count()
print(f"\nChildren per parent — min: {children_per_parent.min()}, "
      f"max: {children_per_parent.max()}, "
      f"mean: {children_per_parent.mean():.2f}")

# Approximate tree depth via BFS
def compute_depths(df):
    parent_map = dict(zip(df["categoryid"], df["parentid"]))
    depths = {}
    for cat in df["categoryid"]:
        depth, node = 0, cat
        visited = set()
        while node in parent_map and not pd.isna(parent_map[node]) and node not in visited:
            visited.add(node)
            node = parent_map[node]
            depth += 1
        depths[cat] = depth
    return depths

depths = compute_depths(category_tree)
depth_series = pd.Series(depths)
print(f"\nTree depth — max: {depth_series.max()}, mean: {depth_series.mean():.2f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
depth_series.value_counts().sort_index().plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Category Count by Depth Level")
axes[0].set_xlabel("Depth")
axes[0].set_ylabel("Count")

children_per_parent.plot(kind="hist", bins=20, ax=axes[1], color="coral", edgecolor="white")
axes[1].set_title("Distribution of Children per Parent Category")
axes[1].set_xlabel("Number of Children")
plt.tight_layout()
plt.show()

## 5. Explore Events

In [ ]:
# Convert timestamp (milliseconds) to datetime
events["datetime"] = pd.to_datetime(events["timestamp"], unit="ms")

print(f"Total events      : {len(events):,}")
print(f"Unique visitors   : {events['visitorid'].nunique():,}")
print(f"Unique items      : {events['itemid'].nunique():,}")
print(f"Date range        : {events['datetime'].min().date()} → {events['datetime'].max().date()}")
print(f"\nEvent type counts:")
print(events["event"].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Event type frequency
events["event"].value_counts().plot(kind="bar", ax=axes[0], color=["#4C72B0","#DD8452","#55A868"])
axes[0].set_title("Event Type Distribution")
axes[0].set_xlabel("Event Type")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=0)

# Events over time (daily)
events.set_index("datetime").resample("D")["event"].count().plot(ax=axes[1], color="steelblue")
axes[1].set_title("Daily Event Volume")
axes[1].set_xlabel("Date")
axes[1].set_ylabel("Number of Events")

plt.tight_layout()
plt.show()

In [ ]:
# Events per visitor distribution
events_per_visitor = events.groupby("visitorid")["event"].count()
print(f"Events per visitor — min: {events_per_visitor.min()}, "
      f"max: {events_per_visitor.max()}, "
      f"median: {events_per_visitor.median()}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

events_per_visitor.clip(upper=50).plot(kind="hist", bins=50, ax=axes[0], color="steelblue", edgecolor="white")
axes[0].set_title("Events per Visitor (clipped at 50)")
axes[0].set_xlabel("Event Count")
axes[0].set_ylabel("Number of Visitors")

# Hour of day activity
events["hour"] = events["datetime"].dt.hour
events.groupby("hour")["event"].count().plot(kind="bar", ax=axes[1], color="coral")
axes[1].set_title("Event Count by Hour of Day")
axes[1].set_xlabel("Hour")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

## 6. Explore Item Properties (Part 1 & Part 2)

In [ ]:
for part_name, df in [("Part 1", item_props_part1), ("Part 2", item_props_part2)]:
    print(f"\n--- Item Properties {part_name} ---")
    print(f"Shape           : {df.shape}")
    print(f"Unique items    : {df['itemid'].nunique():,}")
    print(f"Unique props    : {df['property'].nunique():,}")
    print(f"\nTop 10 properties:")
    print(df["property"].value_counts().head(10))
    print()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, (part_name, df) in zip(axes, [("Part 1", item_props_part1), ("Part 2", item_props_part2)]):
    df["property"].value_counts().head(15).plot(kind="barh", ax=ax, color="steelblue")
    ax.set_title(f"Top 15 Properties — {part_name}")
    ax.set_xlabel("Count")
    ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 7. Merge Item Properties

In [ ]:
item_props_part1["source"] = "part1"
item_props_part2["source"] = "part2"

item_props_all = pd.concat([item_props_part1, item_props_part2], ignore_index=True)

print(f"Combined shape          : {item_props_all.shape}")
print(f"Unique items (combined) : {item_props_all['itemid'].nunique():,}")
print(f"Unique properties       : {item_props_all['property'].nunique():,}")

# Check for duplicates
dup_count = item_props_all.duplicated(subset=["timestamp","itemid","property","value"]).sum()
print(f"\nExact duplicate rows    : {dup_count:,}")

source_counts = item_props_all["source"].value_counts()
source_counts.plot(kind="bar", color=["steelblue","coral"], edgecolor="white")
plt.title("Records per Source File")
plt.xlabel("Source")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 8. Analyze Missing Values

In [ ]:
summary = {}
for name, df in [
    ("category_tree",    category_tree),
    ("events",           events),
    ("item_props_part1", item_props_part1),
    ("item_props_part2", item_props_part2),
    ("item_props_all",   item_props_all),
]:
    miss = df.isnull().sum()
    miss_pct = (miss / len(df) * 100).round(2)
    summary[name] = miss_pct

missing_df = pd.DataFrame(summary).fillna(0)
print("Missing value % per column per dataset:")
display(missing_df)

# Heatmap — only show datasets/columns that actually have nulls
has_nulls = missing_df.loc[(missing_df > 0).any(axis=1)]
if has_nulls.empty:
    print("\nNo missing values found across all datasets.")
else:
    plt.figure(figsize=(12, max(3, len(has_nulls) * 0.5)))
    sns.heatmap(has_nulls, annot=True, fmt=".1f", cmap="YlOrRd", linewidths=0.5)
    plt.title("Missing Value Heatmap (% of rows)")
    plt.tight_layout()
    plt.show()

## 9. Visualize Data Distributions

In [ ]:
# --- Events: event type pie + transactionid null rate ---
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

events["event"].value_counts().plot(kind="pie", ax=axes[0], autopct="%1.1f%%",
                                     colors=["#4C72B0","#DD8452","#55A868"])
axes[0].set_title("Event Type Share")
axes[0].set_ylabel("")

# Items per event type
events.groupby("event")["itemid"].nunique().plot(kind="bar", ax=axes[1], color="steelblue", edgecolor="white")
axes[1].set_title("Unique Items per Event Type")
axes[1].set_xlabel("Event")
axes[1].set_ylabel("Unique Items")
axes[1].tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# --- Item Properties: records per item distribution + property timestamp spread ---
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

records_per_item = item_props_all.groupby("itemid").size()
records_per_item.clip(upper=200).plot(kind="hist", bins=50, ax=axes[0], color="coral", edgecolor="white")
axes[0].set_title("Records per Item (combined, clipped at 200)")
axes[0].set_xlabel("Record Count")
axes[0].set_ylabel("Items")

item_props_all["datetime"] = pd.to_datetime(item_props_all["timestamp"], unit="ms")
item_props_all.set_index("datetime").resample("ME")["itemid"].count().plot(ax=axes[1], marker="o", color="steelblue")
axes[1].set_title("Monthly Item Property Updates")
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Record Count")
plt.tight_layout()
plt.show()

## 10. Explore Relationships Between Datasets

In [ ]:
# --- Events ↔ Item Properties overlap ---
event_items = set(events["itemid"].unique())
prop_items  = set(item_props_all["itemid"].unique())

only_events = len(event_items - prop_items)
only_props  = len(prop_items  - event_items)
both        = len(event_items & prop_items)

print(f"Items in events only           : {only_events:,}")
print(f"Items in item_props only       : {only_props:,}")
print(f"Items present in BOTH datasets : {both:,}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(["Events only", "Both", "Props only"], [only_events, both, only_props],
       color=["#4C72B0","#55A868","#DD8452"])
ax.set_title("Item Overlap between Events and Item Properties")
ax.set_ylabel("Number of Unique Items")
plt.tight_layout()
plt.show()

In [ ]:
# --- Join events with item properties to attach categoryid ---
# Extract categoryid from item_props_all
cat_prop = (
    item_props_all[item_props_all["property"] == "categoryid"]
    [["itemid", "value"]]
    .drop_duplicates(subset="itemid")
    .rename(columns={"value": "categoryid"})
)
cat_prop["categoryid"] = pd.to_numeric(cat_prop["categoryid"], errors="coerce")

events_enriched = events.merge(cat_prop, on="itemid", how="left")
coverage = events_enriched["categoryid"].notna().mean() * 100
print(f"Events with a resolved categoryid : {coverage:.1f}%")

# Top categories by event count
top_cats = (
    events_enriched.dropna(subset=["categoryid"])
    .groupby("categoryid")["event"]
    .count()
    .sort_values(ascending=False)
    .head(15)
)

top_cats.plot(kind="barh", color="steelblue")
plt.gca().invert_yaxis()
plt.title("Top 15 Categories by Event Count")
plt.xlabel("Event Count")
plt.tight_layout()
plt.show()

In [ ]:
# --- Event type breakdown per top category ---
top_cat_ids = top_cats.index.tolist()
subset = events_enriched[events_enriched["categoryid"].isin(top_cat_ids)]

pivot = subset.groupby(["categoryid", "event"]).size().unstack(fill_value=0)
pivot.plot(kind="bar", stacked=True, figsize=(14, 5), colormap="tab10")
plt.title("Event Types Across Top 15 Categories")
plt.xlabel("Category ID")
plt.ylabel("Event Count")
plt.legend(title="Event", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout()
plt.show()